# Paper 02 · Learning Representations by Back-Propagating Errors

**Citation:** D. E. Rumelhart, G. E. Hinton, R. J. Williams, “Learning representations by back-propagating errors” (Nature, 1986).

**Paper:** https://doi.org/10.1038/323533a0

> **Scale gap:** We train a tiny NumPy network on XOR and gradient-check it. This reproduces the mechanism, not the paper's full collection of demonstrations.

## Mathematical Framework

Before reproducing the paper experimentally, work through the relevant mathematical companions:

- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

For the paper defense, be able to explain the **objective, derivation, assumptions, and why the reported mechanism should follow mathematically**, not just what the code did.

## Before you read
1. Why is a hidden layer insufficient without a nonlinearity?
2. What quantity is propagated backward?
3. How would you verify an analytical gradient independently?

## Central claim
Gradient-based error propagation through hidden layers can learn useful internal representations that a single linear unit cannot.

## Partially completed NumPy network

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
from coursekit.experiments import Experiment
experiment = Experiment('paper-02_backpropagation_1986', {'bootstrap_seed': SEED, 'scope': 'educational mechanism demonstration', 'note': 'Original notebook may use additional explicit seeds; source hash records the exact experiment.'}, source='papers/notebooks/02_backpropagation_1986.ipynb')
experiment.capture_figures()

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(1)
X=np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y=np.array([[0.],[1.],[1.],[0.]])
W1=rng.normal(0,.5,(2,4)); b1=np.zeros(4)
W2=rng.normal(0,.5,(4,1)); b2=np.zeros(1)

def sigmoid(z): return 1/(1+np.exp(-z))

def forward(X):
    z1=X@W1+b1
    a1=np.tanh(z1)   # TODO: later replace tanh with identity and explain the result.
    z2=a1@W2+b2
    p=sigmoid(z2)
    return z1,a1,z2,p

## Backpropagation and training

In [ ]:
losses=[]
for step in range(6000):
    z1,a1,z2,p=forward(X)
    loss=-np.mean(y*np.log(p+1e-12)+(1-y)*np.log(1-p+1e-12)); losses.append(loss)
    dz2=(p-y)/len(X)
    dW2=a1.T@dz2; db2=dz2.sum(0)
    da1=dz2@W2.T
    dz1=da1*(1-a1*a1)
    dW1=X.T@dz1; db1=dz1.sum(0)
    lr=.1
    W1-=lr*dW1; b1-=lr*db1; W2-=lr*dW2; b2-=lr*db2
print("predictions:",forward(X)[-1].round(3).ravel())
plt.plot(losses); plt.yscale("log"); plt.title("XOR training loss"); plt.show()

## Gradient check one parameter

In [ ]:
z1,a1,z2,p=forward(X)
dz2=(p-y)/len(X); analytic=(a1.T@dz2)[0,0]
eps=1e-5; original=W2[0,0]
W2[0,0]=original+eps; lp=-np.mean(y*np.log(forward(X)[-1]+1e-12)+(1-y)*np.log(1-forward(X)[-1]+1e-12))
W2[0,0]=original-eps; lm=-np.mean(y*np.log(forward(X)[-1]+1e-12)+(1-y)*np.log(1-forward(X)[-1]+1e-12))
W2[0,0]=original
numeric=(lp-lm)/(2*eps)
print("analytic:",analytic,"numeric:",numeric,"close:",np.isclose(analytic,numeric,rtol=1e-3))

### Ablation
Reinitialize the network and replace `tanh(z1)` with `z1`. Train again. Explain why depth without nonlinearity collapses to a linear map.

## Ablation table
Fill this after running the experiments.

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
Answer without looking back at the notebook:
1. What problem existed before this work?
2. What was actually new?
3. What evidence in your reproduction supports the central claim?
4. What does your reduced-scale reproduction **not** establish?
5. Which idea from this paper survived into modern systems?
6. What experiment would you run next?

## Evidence export

Figures and numeric diagnostics are captured. Explicit metrics use `experiment.log(variant, seed, metrics)`. Use `run_trials` for paired-seed ablations. An empty metrics table or `not_run` ablation is incomplete evidence, not success. Interpretations remain your work.

In [ ]:
print('Evidence directory:', experiment.finish(globals()))